In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import itertools
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from sklearn.base import clone, BaseEstimator, TransformerMixin, check_is_fitted
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, cross_validate, StratifiedKFold
import seaborn as sns
from tqdm.auto import tqdm

from src.data import add_metadata_features
from src.stimuli import POD_dict

In [6]:
epochs_path = "outputs/epochs_preprocessed/EC248_epo.fif"
out_path = "outputs/windowed_regression/EC248/results.pkl"

time_windows = [
    (-0.1, 0.0),
    (0.0, 0.2),
    (0.1, 0.3),
    (0.2, 0.4),
    (0.3, 0.5),
    (0.4, 0.6),
    (0.5, 0.7),
]

features_per_phoneme_pair = [
    ("linear_acoustic_cue", "onset"),
    ("categorical_acoustic_cue", "onset"),
    ("lexical_evidence_cue", "PoD"),
    ("mismatch", "PoD"),
    ("mismatch_left_right", "PoD")
]

# Prepare blocks of features to be removed in UV analysis
feature_blocks = {
    "acoustic": [
        "linear_acoustic_cue",
        "categorical_acoustic_cue",
    ],
    "lexical_evidence": [
        "lexical_evidence_cue",
    ],
    "mismatch": [
        "mismatch",
        "mismatch_left_right",
    ],
}

In [7]:
mne.set_config("MNE_TQDM", "off")

In [ ]:
subject_name = re.findall("(EC[\d]+)_epo", epochs_path)[0]
ep = mne.read_epochs(epochs_path)

In [ ]:
old_metadata = ep.metadata.copy()
ep.metadata = add_metadata_features(old_metadata)

In [11]:
class WindowedEpochRegression(BaseEstimator):
    """
    combines data prep + model fitting in a single estimator
    """
    def __init__(self, estimator, ep: mne.Epochs,
                 tmin: float, tmax: float,
                 features_per_phoneme_pair):
        self.estimator = estimator
        self.ep = ep
        self.tmin, self.tmax = tmin, tmax
        self.features_per_phoneme_pair = features_per_phoneme_pair

        assert self.tmin <= self.tmax
        assert self.tmin >= self.ep.tmin
        assert self.tmax <= self.ep.tmax

        self.idx_min, self.idx_max = self.ep.time_as_index([self.tmin, self.tmax])
        self._update_metadata()

    def _update_metadata(self):
        self.phoneme_pairs = sorted(set(self.ep.metadata.phoneme_pair))
        self.feature_names = [
            f"{feature_name}-{phoneme_pair}"
            for feature_name, _ in self.features_per_phoneme_pair
            for phoneme_pair in self.phoneme_pairs
        ]

    def set_params(self, **params):
        super().set_params(**params)
    
        if "features_per_phoneme_pair" in params or "ep" in params:
            self._update_metadata()

        return self

    def _prepare_design_matrix(self, idxs):
        # HACK this is incorporated here rather than in a Pipeline because we need to
        # modify both X and Y

        X = np.zeros((len(idxs), len(self.feature_names)))
        md = self.ep.metadata

        ep_cropped = self.ep[idxs].copy().crop(tmin=self.tmin, tmax=self.tmax)
        Y = ep_cropped.get_data().mean(axis=2)

        for i, idx in enumerate(idxs):
            md_i = md.iloc[idx]

            # build up design matrix for this trial by column
            j = 0
            for feature_name, feature_alignment in self.features_per_phoneme_pair:
                for phoneme_pair in self.phoneme_pairs:
                    if phoneme_pair == md_i.phoneme_pair:
                        if feature_alignment == "onset":
                            X[i, j] = md_i[feature_name]
                        elif feature_alignment == "PoD":
                            X[i, j] = md_i[feature_name]
                        else:
                            raise ValueError(f"Unknown feature alignment: {feature_alignment}")
                    j += 1

        return X, Y

    def fit(self, idxs, y=None):
        X, Y = self._prepare_design_matrix(idxs)

        est = clone(self.estimator)
        est.feature_names = self.feature_names
        self.estimator_ = est.fit(X, Y)

        return self

    def score_multidimensional(self, idxs, y=None):
        check_is_fitted(self)
        X, Y = self._prepare_design_matrix(idxs)

        # returns one score per output channel
        Y_pred = self.estimator_.predict(X)
        scores = r2_score(Y, Y_pred, multioutput="raw_values")
        return scores
    
    def score(self, idxs, y=None):
        check_is_fitted(self)
        scores = self.score_multidimensional(idxs)

        # take mean across electrodes, ignoring negative results
        scores[scores < 0] = np.nan
        if np.isnan(scores).all():
            return 0
        return np.nanmean(scores)

    @property
    def coef_(self):
        return self.estimator_.coef_
    
    def get_feature_names(self, feature_spec):
        feature_name, _ = feature_spec
        return [f"{feature_name}-{phoneme_pair}" for phoneme_pair in self.phoneme_pairs]

    def predict(self, idxs):
        check_is_fitted(self)
        X, _ = self._prepare_design_matrix(idxs)
        return self.estimator_.predict(X)

In [28]:
def estimate_unique_variance(features_per_phoneme_pair,
                             estimator, train_idxs, test_idxs):
    overall_score = estimator.score_multidimensional(test_idxs)
    
    scored_blocks, scores = [], []
    for feature_block_name, features in feature_blocks.items():
        estimator_modified = clone(estimator)
        estimator_modified.set_params(
            features_per_phoneme_pair=[f for f in features_per_phoneme_pair if f[0] not in features])
        
        estimator_modified.fit(train_idxs)
        score = estimator_modified.score_multidimensional(test_idxs)
        # ignore subzero score
        score[score < 0] = np.nan

        scored_blocks.append(feature_block_name)
        scores.append(score)

    # electrodes with subzero score should not be considered
    reference_score = overall_score.copy()
    reference_score[reference_score < 0] = np.nan
    score_deltas = reference_score[None, :] - np.array(scores)

    return overall_score, scored_blocks, score_deltas


def estimate_windowed(ep: mne.Epochs, tmin, tmax,
                      features_per_phoneme_pair,
                      Cs=None, num_folds=4):
    if Cs is None:
        Cs = np.logspace(-2, 3, 5)
    md = ep.metadata
    epoch_idxs = md.index.values

    # # DEV
    # epoch_idxs = epoch_idxs[:50]
    # num_folds = 2
    # Cs = Cs[:2]

    estimator = Ridge()
    outer_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)

    pipeline = WindowedEpochRegression(
        estimator, ep, tmin=tmin, tmax=tmax,
        features_per_phoneme_pair=features_per_phoneme_pair)
    param_grid = {"estimator__alpha": Cs}

    clf = GridSearchCV(pipeline, param_grid, cv=inner_cv, n_jobs=1)
    stratify_class = md.loc[epoch_idxs].stratify_class

    cv_results = cross_validate(clf, X=epoch_idxs, y=stratify_class,
                                cv=outer_cv, n_jobs=4,
                                return_estimator=True,
                                return_train_score=True,
                                return_indices=True,
                                verbose=100)
    
    best_Cs = [est.best_params_["estimator__alpha"] for est in cv_results["estimator"]]
    picked_C_min = (np.array(best_Cs) == Cs[0]).mean() * 100
    picked_C_max = (np.array(best_Cs) == Cs[-1]).mean() * 100
    print(f"picked C_min: {picked_C_min:.2f}%, picked C_max: {picked_C_max:.2f}%")

    # Estimate unique variance explained per feature on the outer folds.
    overall_scores, unique_variance_estimates = [], []
    for gs, train_idxs, test_idxs in zip(tqdm(cv_results["estimator"], desc="Estimate unique variance", unit="fold", leave=False),
                                         cv_results["indices"]["train"],
                                         cv_results["indices"]["test"]):
        overall_score, block_names, per_block_uv = estimate_unique_variance(
            features_per_phoneme_pair, gs.best_estimator_, train_idxs, test_idxs
        )
        overall_scores.append(overall_score)
        unique_variance_estimates.append(dict(zip(block_names, per_block_uv)))

    overall_scores = pd.DataFrame(overall_scores)
    overall_scores.index.name = "fold"
    overall_scores.columns.name = "electrode"
    unique_variance_estimates = pd.concat({
            fold: pd.DataFrame.from_dict(fold_uv_estimates, orient="index")
            for fold, fold_uv_estimates in enumerate(unique_variance_estimates)
        }, names=["fold", "feature_block"])
    unique_variance_estimates.columns.name = "electrode"
    
    return cv_results, overall_scores, unique_variance_estimates

In [ ]:
cv_results, overall_scores, unique_variance_estimates = estimate_windowed(
    ep, tmin=-0.1, tmax=0.0, features_per_phoneme_pair=features_per_phoneme_pair)

In [ ]:
all_results = {
    "windows": time_windows,
    "results": [],
    "feature_blocks": feature_blocks,
}
Cs = np.logspace(-4, 2, 5)

for tmin, tmax in tqdm(time_windows, desc="Time windows"):
    cv_results, overall_scores, unique_variance_estimates = estimate_windowed(
        ep, tmin, tmax, Cs=Cs, features_per_phoneme_pair=features_per_phoneme_pair)
    all_results["results"].append({
        "tmin": tmin,
        "tmax": tmax,

        "unique_variance_df": unique_variance_estimates,
        "overall_scores_df": overall_scores,

        "C_space": Cs,
        "Cs": [gs.best_params_["estimator__alpha"] for gs in cv_results["estimator"]],

        "estimators" : [gs.best_estimator_.estimator_ for gs in cv_results["estimator"]],
        "train_score": cv_results["train_score"],
        "test_score": cv_results["test_score"],
    })

In [109]:
with open(out_path, "wb") as f:
    pickle.dump(all_results, f)